# SentinelFlow AI — Feature Engineering & ML Preparation

## Objective

This notebook prepares the cleaned CIC-IDS2017 dataset for machine learning.

The main objectives are:

- Load and inspect the processed dataset
- Verify feature data types
- Identify constant and non-informative features
- Analyze feature quality and distributions
- Identify potential data leakage
- Prepare the target variable
- Select appropriate features for model training
- Prepare the dataset for train, validation, and test splitting

## Guiding Principle

Feature decisions will be based on analysis rather than removing or transforming features blindly.

The processed dataset will remain unchanged, and all machine-learning transformations will be performed separately.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("data/processed")

processed_files = sorted(PROCESSED_DIR.glob("*.csv"))

print(f"Processed files found: {len(processed_files)}")

for file in processed_files:
    print(file.name)

Processed files found: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [2]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
C:\Users\abdal\SentinelFlow-AI


In [4]:
sample_df = pd.read_csv(
    processed_files[0],
    nrows=5000,
    low_memory=False
)

print(f"Sample shape: {sample_df.shape}")
sample_df.head()

Sample shape: (5000, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
dtype_summary = pd.DataFrame({
    "column": sample_df.columns,
    "dtype": sample_df.dtypes.astype(str).values,
    "unique_values": [sample_df[col].nunique() for col in sample_df.columns]
})

print("Data type distribution:")
print(dtype_summary["dtype"].value_counts())

dtype_summary

Data type distribution:
dtype
int64      54
float64    24
str         1
Name: count, dtype: int64


,column,dtype,unique_values
0,Destination Port,int64,642
1,Flow Duration,int64,3744
2,Total Fwd Packets,int64,85
3,Total Backward Packets,int64,100
4,Total Length of Fwd Packets,int64,892
...,...,...,...
74,Idle Mean,float64,433
75,Idle Std,float64,488
76,Idle Max,int64,274
77,Idle Min,int64,554


In [6]:
sample_constant_features = [
    col for col in sample_df.columns
    if col != "Label" and sample_df[col].nunique(dropna=False) <= 1
]

print(f"Potential constant features in sample: {len(sample_constant_features)}")

for col in sample_constant_features:
    print(col)

Potential constant features in sample: 12
Bwd PSH Flags
Fwd URG Flags
Bwd URG Flags
RST Flag Count
CWE Flag Count
ECE Flag Count
Fwd Avg Bytes/Bulk
Fwd Avg Packets/Bulk
Fwd Avg Bulk Rate
Bwd Avg Bytes/Bulk
Bwd Avg Packets/Bulk
Bwd Avg Bulk Rate


## Full-Dataset Constant Feature Analysis

Features that appear constant in a small sample are not removed immediately.

To avoid making decisions based on an unrepresentative sample, all processed files are scanned to determine which features are truly constant across the entire dataset.

A feature is considered constant only if it contains a single unique value across all processed network flows.

In [7]:
feature_unique_values = {
    col: set()
    for col in sample_df.columns
    if col != "Label"
}

for file in processed_files:
    print(f"Scanning: {file.name}")

    for chunk in pd.read_csv(file, chunksize=100000, low_memory=False):
        for col in feature_unique_values:
            feature_unique_values[col].update(chunk[col].unique())

print("Scan complete.")

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
Scanning: Monday-WorkingHours.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
Scanning: Wednesday-workingHours.pcap_ISCX.csv
Scan complete.


In [8]:
constant_features = [
    col for col, values in feature_unique_values.items()
    if len(values) == 1
]

print(f"Confirmed constant features: {len(constant_features)}")

for col in constant_features:
    value = next(iter(feature_unique_values[col]))
    print(f"{col}: {value}")

Confirmed constant features: 8
Bwd PSH Flags: 0
Bwd URG Flags: 0
Fwd Avg Bytes/Bulk: 0
Fwd Avg Packets/Bulk: 0
Fwd Avg Bulk Rate: 0
Bwd Avg Bytes/Bulk: 0
Bwd Avg Packets/Bulk: 0
Bwd Avg Bulk Rate: 0


### Constant Feature Decision

A full scan of all 2,572,640 processed network flows identified 8 features that are constant across the entire dataset.

All 8 features contain only the value `0`, meaning they have zero variance and provide no discriminative information for machine learning.

These features will therefore be excluded from the machine-learning feature set:

- Bwd PSH Flags
- Bwd URG Flags
- Fwd Avg Bytes/Bulk
- Fwd Avg Packets/Bulk
- Fwd Avg Bulk Rate
- Bwd Avg Bytes/Bulk
- Bwd Avg Packets/Bulk
- Bwd Avg Bulk Rate

The original processed dataset remains unchanged. Feature removal will be applied only when constructing the machine-learning dataset.

## Duplicate Feature Analysis

Different feature names may occasionally contain identical information.

As an initial screening step, the sample dataset is analyzed for features with identical values. Any potential duplicate features identified from the sample will later be verified using the full processed dataset before any feature is excluded.

In [9]:
candidate_features = [
    col for col in sample_df.columns
    if col != "Label" and col not in constant_features
]

duplicate_feature_pairs = []

for i, col1 in enumerate(candidate_features):
    for col2 in candidate_features[i + 1:]:
        if sample_df[col1].equals(sample_df[col2]):
            duplicate_feature_pairs.append((col1, col2))

print(f"Potential duplicate feature pairs in sample: {len(duplicate_feature_pairs)}")

for col1, col2 in duplicate_feature_pairs:
    print(f"{col1}  <->  {col2}")

Potential duplicate feature pairs in sample: 14
Total Fwd Packets  <->  Subflow Fwd Packets
Total Backward Packets  <->  Subflow Bwd Packets
Total Length of Fwd Packets  <->  Subflow Fwd Bytes
Total Length of Bwd Packets  <->  Subflow Bwd Bytes
Fwd Packet Length Mean  <->  Avg Fwd Segment Size
Bwd Packet Length Mean  <->  Avg Bwd Segment Size
Fwd PSH Flags  <->  SYN Flag Count
Fwd URG Flags  <->  RST Flag Count
Fwd URG Flags  <->  CWE Flag Count
Fwd URG Flags  <->  ECE Flag Count
Fwd Header Length  <->  Fwd Header Length.1
RST Flag Count  <->  CWE Flag Count
RST Flag Count  <->  ECE Flag Count
CWE Flag Count  <->  ECE Flag Count


### Full-Dataset Duplicate Feature Verification

The duplicate feature pairs detected in the sample are only candidates.

Each candidate pair is verified across all processed network flows before any feature is considered redundant. Two features are classified as duplicates only if their values are identical for every row in every processed file.

In [10]:
confirmed_duplicate_pairs = []

for col1, col2 in duplicate_feature_pairs:
    is_duplicate = True

    for file in processed_files:
        for chunk in pd.read_csv(
            file,
            usecols=[col1, col2],
            chunksize=100000,
            low_memory=False
        ):
            if not chunk[col1].equals(chunk[col2]):
                is_duplicate = False
                break

        if not is_duplicate:
            break

    if is_duplicate:
        confirmed_duplicate_pairs.append((col1, col2))

print(f"Confirmed duplicate feature pairs: {len(confirmed_duplicate_pairs)}")

for col1, col2 in confirmed_duplicate_pairs:
    print(f"{col1}  <->  {col2}")

Confirmed duplicate feature pairs: 5
Total Fwd Packets  <->  Subflow Fwd Packets
Total Backward Packets  <->  Subflow Bwd Packets
Fwd PSH Flags  <->  SYN Flag Count
Fwd URG Flags  <->  CWE Flag Count
Fwd Header Length  <->  Fwd Header Length.1


### Duplicate Feature Decision

Full-dataset verification confirmed 5 feature pairs with identical values across all 2,572,640 processed network flows.

To avoid redundant input features, one feature from each confirmed duplicate pair will be excluded from the machine-learning feature set.

Features selected for exclusion:

- Subflow Fwd Packets
- Subflow Bwd Packets
- SYN Flag Count
- CWE Flag Count
- Fwd Header Length.1

The corresponding retained features are:

- Total Fwd Packets
- Total Backward Packets
- Fwd PSH Flags
- Fwd URG Flags
- Fwd Header Length

The TCP flag pairs are treated as empirically redundant in this dataset because their observed values are identical across all processed flows; this does not imply that the underlying network concepts are equivalent.

As with constant-feature removal, the processed dataset itself remains unchanged. These exclusions will only be applied when constructing the machine-learning feature set.

In [11]:
duplicate_features_to_remove = [
    "Subflow Fwd Packets",
    "Subflow Bwd Packets",
    "SYN Flag Count",
    "CWE Flag Count",
    "Fwd Header Length.1"
]

candidate_features = [
    col for col in sample_df.columns
    if col != "Label"
    and col not in constant_features
    and col not in duplicate_features_to_remove
]

print(f"Candidate features remaining: {len(candidate_features)}")

for i, col in enumerate(candidate_features, start=1):
    print(f"{i:02d}. {col}")

Candidate features remaining: 65
01. Destination Port
02. Flow Duration
03. Total Fwd Packets
04. Total Backward Packets
05. Total Length of Fwd Packets
06. Total Length of Bwd Packets
07. Fwd Packet Length Max
08. Fwd Packet Length Min
09. Fwd Packet Length Mean
10. Fwd Packet Length Std
11. Bwd Packet Length Max
12. Bwd Packet Length Min
13. Bwd Packet Length Mean
14. Bwd Packet Length Std
15. Flow Bytes/s
16. Flow Packets/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Total
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Total
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Fwd URG Flags
33. Fwd Header Length
34. Bwd Header Length
35. Fwd Packets/s
36. Bwd Packets/s
37. Min Packet Length
38. Max Packet Length
39. Packet Length Mean
40. Packet Length Std
41. Packet Length Variance
42. FIN Flag Count
43. RST Flag Count
44. PSH Flag Count
45. ACK Flag Count
46. URG Flag Count
47. ECE

## Data Leakage Screening

Potential data leakage cannot be identified reliably from feature names alone.

As an initial screening step, a representative sample is collected across all processed files. The relationship between candidate features and the target label will then be examined for suspiciously direct target information.

Features will not be removed solely because they are strongly associated with the target. Strong predictive relationships may be legitimate in intrusion-detection data and must be distinguished from actual leakage.

In [12]:
samples = []

for file in processed_files:
    df_part = pd.read_csv(file, low_memory=False)

    sample_size = min(10000, len(df_part))

    df_part = df_part.sample(
        n=sample_size,
        random_state=42
    )

    samples.append(df_part)

    del df_part

leakage_sample = pd.concat(samples, ignore_index=True)

print(f"Leakage analysis sample shape: {leakage_sample.shape}")
print(f"Unique labels: {leakage_sample['Label'].nunique()}")

print("\nLabel distribution:")
print(leakage_sample["Label"].value_counts())

Leakage analysis sample shape: (80000, 79)
Unique labels: 14

Label distribution:
Label
BENIGN                      66418
DDoS                         5746
PortScan                     4316
DoS Hulk                     2764
DoS GoldenEye                 177
FTP-Patator                   123
DoS slowloris                  92
Bot                            91
Web Attack Brute Force         87
DoS Slowhttptest               76
SSH-Patator                    69
Web Attack XSS                 38
Infiltration                    2
Web Attack Sql Injection        1
Name: count, dtype: int64


### Stratified Leakage Analysis Sample

The initial random sample contained only 14 of the 15 traffic classes and severely underrepresented several rare attacks.

To ensure that leakage analysis considers every traffic class, a class-aware sample is constructed across the complete processed dataset.

Rare classes are preserved, while larger classes are sampled to keep the analysis computationally manageable.

In [13]:
all_parts = []

for file in processed_files:
    df_part = pd.read_csv(file, low_memory=False)
    all_parts.append(df_part)

full_df = pd.concat(all_parts, ignore_index=True)

print(f"Full processed dataset shape: {full_df.shape}")
print(f"Unique labels: {full_df['Label'].nunique()}")

Full processed dataset shape: (2572640, 79)
Unique labels: 15


In [14]:
MAX_SAMPLES_PER_CLASS = 5000

stratified_parts = []

for label, group in full_df.groupby("Label"):
    sample_size = min(MAX_SAMPLES_PER_CLASS, len(group))

    sampled_group = group.sample(
        n=sample_size,
        random_state=42
    )

    stratified_parts.append(sampled_group)

stratified_sample = pd.concat(
    stratified_parts,
    ignore_index=True
)

print(f"Stratified sample shape: {stratified_sample.shape}")
print(f"Unique labels: {stratified_sample['Label'].nunique()}")

print("\nLabel distribution:")
print(stratified_sample["Label"].value_counts())

Stratified sample shape: (47357, 79)
Unique labels: 15

Label distribution:
Label
BENIGN                      5000
DDoS                        5000
DoS GoldenEye               5000
DoS Hulk                    5000
DoS Slowhttptest            5000
DoS slowloris               5000
FTP-Patator                 5000
PortScan                    5000
SSH-Patator                 3219
Bot                         1948
Web Attack Brute Force      1470
Web Attack XSS               652
Infiltration                  36
Web Attack Sql Injection      21
Heartbleed                    11
Name: count, dtype: int64


In [15]:
del full_df
del all_parts

import gc
gc.collect()

print("Full dataset released from memory.")
print(f"Analysis sample retained: {stratified_sample.shape}")

Full dataset released from memory.
Analysis sample retained: (47357, 79)


### Feature-to-Target Purity Screening

A highly predictive feature is not automatically considered data leakage.

For features with a manageable number of unique values, this analysis measures how strongly individual feature values correspond to a single traffic class.

Features showing unusually high target purity will be flagged for further investigation rather than removed automatically.

In [16]:
purity_results = []

for col in candidate_features:
    unique_count = stratified_sample[col].nunique()

    # Purity analysis is most interpretable for
    # features with a limited number of unique values.
    if unique_count <= 100:
        value_label_counts = pd.crosstab(
            stratified_sample[col],
            stratified_sample["Label"]
        )

        dominant_counts = value_label_counts.max(axis=1)
        total_counts = value_label_counts.sum(axis=1)

        weighted_purity = (
            dominant_counts.sum() / total_counts.sum()
        )

        purity_results.append({
            "feature": col,
            "unique_values": unique_count,
            "target_purity": weighted_purity
        })

purity_df = pd.DataFrame(purity_results).sort_values(
    "target_purity",
    ascending=False
)

print(f"Features analyzed: {len(purity_df)}")
purity_df

Features analyzed: 13


,feature,unique_values,target_purity
11,act_data_pkt_fwd,87,0.445320
12,min_seg_size_forward,9,0.330574
0,Fwd Packet Length Min,97,0.277741
3,Min Packet Length,83,0.251283
10,Down/Up Ratio,8,0.225056
6,PSH Flag Count,2,0.201850
7,ACK Flag Count,2,0.165720
4,FIN Flag Count,2,0.141816
1,Fwd PSH Flags,2,0.139874
8,URG Flag Count,2,0.120426


### Leakage Screening Result

The feature-to-target purity analysis examined 13 candidate features with at most 100 unique values.

No feature showed near-perfect correspondence with the target label. The highest observed target purity was approximately 0.445 for `act_data_pkt_fwd`, which does not indicate an obvious direct encoding of the target.

Therefore, no additional feature is removed based on this screening step.

Strong feature-target relationships, if identified later during modeling, will be investigated carefully to distinguish legitimate network behavior from potential data leakage.

## Feature Correlation Analysis

After removing constant and exactly duplicated features, the remaining candidate features are analyzed for strong numerical correlations.

Highly correlated features may contain redundant information even when their values are not exactly identical.

This analysis is used as a diagnostic step. Features will not be removed automatically based on correlation alone; highly correlated pairs will first be inspected and interpreted.

In [17]:
correlation_matrix = stratified_sample[candidate_features].corr(
    method="pearson"
)

high_correlation_pairs = []

for i in range(len(candidate_features)):
    for j in range(i + 1, len(candidate_features)):

        correlation = correlation_matrix.iloc[i, j]

        if abs(correlation) >= 0.95:
            high_correlation_pairs.append({
                "feature_1": candidate_features[i],
                "feature_2": candidate_features[j],
                "correlation": correlation
            })

high_corr_df = pd.DataFrame(high_correlation_pairs)

if not high_corr_df.empty:
    high_corr_df["abs_correlation"] = high_corr_df["correlation"].abs()

    high_corr_df = high_corr_df.sort_values(
        "abs_correlation",
        ascending=False
    ).drop(columns="abs_correlation")

print(
    f"Highly correlated feature pairs (|r| >= 0.95): "
    f"{len(high_corr_df)}"
)

high_corr_df

Highly correlated feature pairs (|r| >= 0.95): 36


,feature_1,feature_2,correlation
33,RST Flag Count,ECE Flag Count,1.000000
16,Bwd Packet Length Mean,Avg Bwd Segment Size,1.000000
7,Fwd Packet Length Mean,Avg Fwd Segment Size,1.000000
5,Total Length of Fwd Packets,Subflow Fwd Bytes,1.000000
6,Total Length of Bwd Packets,Subflow Bwd Bytes,1.000000
21,Flow IAT Max,Fwd IAT Max,0.999481
0,Flow Duration,Fwd IAT Total,0.998935
29,Packet Length Mean,Average Packet Size,0.998627
9,Bwd Packet Length Max,Bwd Packet Length Std,0.989340
25,Fwd IAT Max,Idle Max,0.988679


### Verification of Perfectly Correlated Features

Several feature pairs showed a Pearson correlation of exactly 1.0 in the stratified analysis sample.

Perfect correlation does not necessarily imply identical values. Therefore, these pairs are verified across the complete processed dataset before being classified as exact duplicates or merely perfectly linearly related.

In [18]:
perfect_corr_pairs = high_corr_df[
    np.isclose(high_corr_df["correlation"].abs(), 1.0)
][["feature_1", "feature_2"]].values.tolist()

perfect_pair_results = []

for col1, col2 in perfect_corr_pairs:
    identical = True
    max_abs_difference = 0.0

    for file in processed_files:
        for chunk in pd.read_csv(
            file,
            usecols=[col1, col2],
            chunksize=100000,
            low_memory=False
        ):
            values1 = chunk[col1].to_numpy()
            values2 = chunk[col2].to_numpy()

            if not np.array_equal(values1, values2):
                identical = False

                difference = np.max(
                    np.abs(values1.astype(float) - values2.astype(float))
                )

                max_abs_difference = max(
                    max_abs_difference,
                    difference
                )

    perfect_pair_results.append({
        "feature_1": col1,
        "feature_2": col2,
        "identical_across_full_dataset": identical,
        "max_absolute_difference": max_abs_difference
    })

perfect_pair_df = pd.DataFrame(perfect_pair_results)

perfect_pair_df

,feature_1,feature_2,identical_across_full_dataset,max_absolute_difference
0,RST Flag Count,ECE Flag Count,False,1.000000e+00
1,Bwd Packet Length Mean,Avg Bwd Segment Size,False,1.000000e-06
2,Fwd Packet Length Mean,Avg Fwd Segment Size,False,9.993073e-11
3,Total Length of Fwd Packets,Subflow Fwd Bytes,False,2.966200e+04
4,Total Length of Bwd Packets,Subflow Bwd Bytes,False,4.904610e+05


### Correlation Analysis Result

The correlation analysis identified 36 feature pairs with an absolute Pearson correlation of at least 0.95.

Five pairs showed a correlation of approximately 1.0 in the stratified analysis sample. These pairs were subsequently verified across the complete processed dataset.

None of the five pairs were exactly identical across the full dataset. Therefore, they are treated as highly correlated features rather than exact duplicates.

No additional features are removed based solely on correlation at this stage.

The current machine-learning candidate feature set therefore remains at 65 features. Correlation will be considered later together with model type, validation performance, and feature importance.

## Feature Distribution Analysis

The remaining 65 candidate features are analyzed to understand their numerical distributions and scales.

This analysis focuses on:

- Value ranges
- Central tendency
- Variability
- Skewness
- Potential extreme values

Large differences in feature scales or highly skewed distributions may affect some machine-learning algorithms. No transformation is applied at this stage; the results are first analyzed to determine whether scaling or other transformations are necessary.

In [19]:
distribution_summary = pd.DataFrame({
    "min": stratified_sample[candidate_features].min(),
    "median": stratified_sample[candidate_features].median(),
    "mean": stratified_sample[candidate_features].mean(),
    "max": stratified_sample[candidate_features].max(),
    "std": stratified_sample[candidate_features].std(),
    "skewness": stratified_sample[candidate_features].skew()
})

distribution_summary["range"] = (
    distribution_summary["max"] - distribution_summary["min"]
)

distribution_summary["abs_skewness"] = (
    distribution_summary["skewness"].abs()
)

distribution_summary = distribution_summary.sort_values(
    "abs_skewness",
    ascending=False
)

print(f"Features analyzed: {len(distribution_summary)}")

distribution_summary

Features analyzed: 65


,min,median,mean,max,std,skewness,range,abs_skewness
RST Flag Count,0.0,0.000000,0.000021,1.000000e+00,4.595237e-03,217.616635,1.000000e+00,217.616635
ECE Flag Count,0.0,0.000000,0.000021,1.000000e+00,4.595237e-03,217.616635,1.000000e+00,217.616635
Flow Bytes/s,0.0,141.467869,333115.276468,1.980000e+09,1.328777e+07,102.813401,1.980000e+09,102.813401
Subflow Fwd Bytes,0.0,48.000000,909.444918,2.866110e+06,2.324182e+04,90.016703,2.866110e+06,90.016703
Total Length of Fwd Packets,0.0,48.000000,909.444918,2.866110e+06,2.324182e+04,90.016703,2.866110e+06,90.016703
...,...,...,...,...,...,...,...,...
ACK Flag Count,0.0,0.000000,0.287455,1.000000e+00,4.525803e-01,0.939299,1.000000e+00,0.939299
PSH Flag Count,0.0,1.000000,0.628735,1.000000e+00,4.831483e-01,-0.532923,1.000000e+00,0.532923
min_seg_size_forward,0.0,32.000000,29.731529,5.600000e+01,6.565128e+00,-0.268081,5.600000e+01,0.268081
Init_Win_bytes_forward,-1.0,8192.000000,15364.071162,6.553500e+04,1.427230e+04,0.120769,6.553600e+04,0.120769


In [20]:
top_skewed_features = distribution_summary[
    [
        "min",
        "median",
        "mean",
        "max",
        "skewness"
    ]
].head(20)

top_skewed_features

,min,median,mean,max,skewness
RST Flag Count,0.000000,0.000000,0.000021,1.000000e+00,217.616635
ECE Flag Count,0.000000,0.000000,0.000021,1.000000e+00,217.616635
Flow Bytes/s,0.000000,141.467869,333115.276468,1.980000e+09,102.813401
Subflow Fwd Bytes,0.000000,48.000000,909.444918,2.866110e+06,90.016703
Total Length of Fwd Packets,0.000000,48.000000,909.444918,2.866110e+06,90.016703
act_data_pkt_fwd,0.000000,1.000000,3.937327,5.522000e+03,78.366171
Subflow Bwd Bytes,0.000000,6.000000,5401.425301,1.181199e+07,65.872586
Total Length of Bwd Packets,0.000000,6.000000,5401.172097,1.180000e+07,65.857965
Total Backward Packets,0.000000,2.000000,7.069578,7.292000e+03,65.794927
Bwd Header Length,0.000000,64.000000,210.309395,1.458400e+05,53.929075


### Distribution Analysis Result

The 65 candidate features show substantial differences in both scale and distribution.

Several continuous and count-based features are strongly right-skewed. For example, `Flow Bytes/s` has a median of approximately 141.47 but reaches values as high as 1.98 billion, resulting in a skewness above 100.

Packet-count, byte-count, header-length, and packet-rate features also show long-tailed distributions with relatively small typical values and much larger extreme observations.

Binary flag features can also exhibit very high skewness when one value is rare. Such skewness does not by itself justify transforming these features.

Therefore, no global transformation is applied blindly. Scaling and possible transformations will be incorporated into the machine-learning pipeline when appropriate for the selected model and will be fitted using training data only to prevent data leakage.

## Target Preparation

SentinelFlow AI will approach intrusion analysis as a two-stage machine-learning problem.

### Stage 1 — Binary Intrusion Detection

The first model will determine whether a network flow is benign or malicious:

- `0` — BENIGN
- `1` — ATTACK

All attack categories are grouped into the malicious class for this stage.

### Stage 2 — Multiclass Attack Classification

A second modeling stage will investigate classification of malicious traffic into its specific attack category.

This two-stage design separates the fundamental intrusion-detection problem from the more difficult task of identifying the exact attack type.

The original `Label` column will be preserved, while a separate binary target will be created for Stage 1.

In [22]:
binary_counts = {
    "BENIGN": 0,
    "ATTACK": 0
}

for file in processed_files:
    for chunk in pd.read_csv(
        file,
        usecols=["Label"],
        chunksize=100000,
        low_memory=False
    ):
        binary_counts["BENIGN"] += (chunk["Label"] == "BENIGN").sum()
        binary_counts["ATTACK"] += (chunk["Label"] != "BENIGN").sum()

total_flows = sum(binary_counts.values())

print("Binary target distribution:")
print(f"BENIGN (0): {binary_counts['BENIGN']:,} "
      f"({binary_counts['BENIGN'] / total_flows * 100:.2f}%)")

print(f"ATTACK (1): {binary_counts['ATTACK']:,} "
      f"({binary_counts['ATTACK'] / total_flows * 100:.2f}%)")

print(f"\nTotal: {total_flows:,}")

Binary target distribution:
BENIGN (0): 2,146,899 (83.45%)
ATTACK (1): 425,741 (16.55%)

Total: 2,572,640


### Binary Target Distribution

Across the complete processed dataset of 2,572,640 network flows:

- **BENIGN (0):** 2,146,899 flows — 83.45%
- **ATTACK (1):** 425,741 flows — 16.55%

The binary intrusion-detection problem therefore exhibits class imbalance, with benign traffic substantially more common than malicious traffic.

Because of this imbalance, model evaluation will not rely on accuracy alone. Metrics such as precision, recall, F1-score, and the confusion matrix will be used to evaluate intrusion-detection performance.

Class-imbalance handling strategies will be considered during model training and will be applied only to the training data when necessary.

## Train / Validation / Test Split Strategy

Before splitting the dataset, the distribution of traffic classes across the processed source files is examined.

CIC-IDS2017 contains traffic collected across different capture periods, and attack categories are not necessarily distributed uniformly across all files.

Therefore, the split strategy will be selected only after analyzing class coverage across the source files.

The goal is to reduce data leakage while maintaining meaningful class representation in the training, validation, and test sets.

In [23]:
file_class_distribution = {}

for file in processed_files:
    label_counts = {}

    for chunk in pd.read_csv(
        file,
        usecols=["Label"],
        chunksize=100000,
        low_memory=False
    ):
        counts = chunk["Label"].value_counts()

        for label, count in counts.items():
            label_counts[label] = (
                label_counts.get(label, 0) + count
            )

    file_class_distribution[file.name] = label_counts

file_class_df = (
    pd.DataFrame(file_class_distribution)
    .fillna(0)
    .astype(int)
)

file_class_df

,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,Friday-WorkingHours-Morning.pcap_ISCX.csv,Monday-WorkingHours.pcap_ISCX.csv,Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,Tuesday-WorkingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv
DDoS,128014,0,0,0,0,0,0,0
BENIGN,95068,123083,182096,502650,252754,162036,412476,416736
PortScan,0,90694,0,0,0,0,0,0
Bot,0,0,1948,0,0,0,0,0
Infiltration,0,0,0,0,36,0,0,0
Web Attack Brute Force,0,0,0,0,0,1470,0,0
Web Attack XSS,0,0,0,0,0,652,0,0
Web Attack Sql Injection,0,0,0,0,0,21,0,0
FTP-Patator,0,0,0,0,0,0,5931,0
SSH-Patator,0,0,0,0,0,0,3219,0


In [24]:
for file in file_class_df.columns:
    print(f"\n{'=' * 70}")
    print(file)
    print("=" * 70)

    counts = file_class_df[file]
    counts = counts[counts > 0].sort_values(ascending=False)

    print(counts)


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
DDoS      128014
BENIGN     95068
Name: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv, dtype: int64

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
BENIGN      123083
PortScan     90694
Name: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX.csv
BENIGN    182096
Bot         1948
Name: Friday-WorkingHours-Morning.pcap_ISCX.csv, dtype: int64

Monday-WorkingHours.pcap_ISCX.csv
BENIGN    502650
Name: Monday-WorkingHours.pcap_ISCX.csv, dtype: int64

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
BENIGN          252754
Infiltration        36
Name: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
BENIGN                      162036
Web Attack Brute Force        1470
Web Attack XSS                 652
Web Attack Sql Injection        21
Name: Thursday-WorkingHours-Morning-WebAttacks.pcap_IS

### Source-File and Class Distribution

The analysis shows that attack categories are strongly associated with specific CIC-IDS2017 source files and capture periods.

For example:

- Friday contains DDoS, PortScan, and Bot traffic.
- Tuesday contains FTP-Patator and SSH-Patator attacks.
- Wednesday contains the DoS attack families and Heartbleed.
- Thursday contains Web Attacks and Infiltration.
- Monday contains only benign traffic.

A strict source-file-based split would therefore cause some attack categories to appear exclusively in either training or evaluation data.

For the primary model-development split, a stratified strategy will be used to preserve class representation across the training, validation, and test sets.

A separate source/day-based evaluation can later be used as a more challenging stress test for temporal and scenario generalization.

All preprocessing operations that learn parameters from the data, including scaling or resampling, will be fitted on the training set only.

In [25]:
split_preview = []

for label in file_class_df.index:
    total = int(file_class_df.loc[label].sum())

    train_count = round(total * 0.70)
    validation_count = round(total * 0.15)
    test_count = total - train_count - validation_count

    split_preview.append({
        "Label": label,
        "Total": total,
        "Train_70%": train_count,
        "Validation_15%": validation_count,
        "Test_15%": test_count
    })

split_preview_df = pd.DataFrame(split_preview).sort_values(
    "Total",
    ascending=False
)

split_preview_df

,Label,Total,Train_70%,Validation_15%,Test_15%
1,BENIGN,2146899,1502829,322035,322035
10,DoS Hulk,172846,120992,25927,25927
0,DDoS,128014,89610,19202,19202
2,PortScan,90694,63486,13604,13604
13,DoS GoldenEye,10286,7200,1543,1543
8,FTP-Patator,5931,4152,890,889
11,DoS slowloris,5385,3769,808,808
12,DoS Slowhttptest,5228,3660,784,784
9,SSH-Patator,3219,2253,483,483
3,Bot,1948,1364,292,292


### Split Ratio and Rare-Class Considerations

The primary dataset split will use:

- **70% Training**
- **15% Validation**
- **15% Testing**

Stratification will be used to preserve the class distribution as closely as possible across the three subsets.

Most attack categories contain enough observations for meaningful representation across all subsets. However, several classes are extremely rare:

- `Infiltration`: 36 flows
- `Web Attack Sql Injection`: 21 flows
- `Heartbleed`: 11 flows

These rare classes will be retained rather than removed. However, class-specific evaluation metrics for these categories must be interpreted cautiously because the validation and test sets contain very few observations.

Overall multiclass performance will therefore be evaluated using aggregate metrics together with per-class results and support counts.

The test set will remain untouched during model development and will only be used for final evaluation.

## Final Feature Set for Machine Learning

After full-dataset analysis:

- 8 zero-variance features were excluded.
- 5 exact duplicate features were excluded.
- No additional features were removed based solely on correlation or target-purity screening.

The final machine-learning candidate set therefore contains **65 input features**.

The original `Label` column is preserved as the multiclass target.

For binary intrusion detection, a derived target will be created:

- `0` — BENIGN
- `1` — ATTACK

The processed source files remain unchanged. Feature selection and target construction are performed separately for machine-learning preparation.

In [26]:
selected_features = candidate_features.copy()

target_columns = [
    "Label",
    "binary_target"
]

print(f"Selected ML features: {len(selected_features)}")
print(f"Target columns: {target_columns}")

print("\nSelected features:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i:2}. {feature}")

Selected ML features: 65
Target columns: ['Label', 'binary_target']

Selected features:
 1. Destination Port
 2. Flow Duration
 3. Total Fwd Packets
 4. Total Backward Packets
 5. Total Length of Fwd Packets
 6. Total Length of Bwd Packets
 7. Fwd Packet Length Max
 8. Fwd Packet Length Min
 9. Fwd Packet Length Mean
10. Fwd Packet Length Std
11. Bwd Packet Length Max
12. Bwd Packet Length Min
13. Bwd Packet Length Mean
14. Bwd Packet Length Std
15. Flow Bytes/s
16. Flow Packets/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Total
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Total
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Fwd URG Flags
33. Fwd Header Length
34. Bwd Header Length
35. Fwd Packets/s
36. Bwd Packets/s
37. Min Packet Length
38. Max Packet Length
39. Packet Length Mean
40. Packet Length Std
41. Packet Length Variance
42. FIN Flag Count
43. RST Flag Count
44. PSH F

In [1]:
import sklearn

print(f"scikit-learn version: {sklearn.__version__}")

ImportError: DLL load failed while importing _csparsetools: An Application Control policy has blocked this file.

In [28]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 2.9 MB/s eta 0:00:03
   ---------- ----------------------------- 2.1/8.2 MB 5.8 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.2 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.2 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 9.9 MB/s  0:00:01
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---- ----------------------------------- 4.5/36.6 MB 20.6 MB/s eta 0:00:02
   ---------- ----------------------------- 9.4/36.6 MB 21.8 MB/s eta 0:00:02
   --------------- ------------------------ 13.9/36.6 MB 21.6 MB/s eta 0:00:02
   -------------------- ------------------- 18.6/36.6 MB 22.0 MB/s eta 0:00:01
   -------------------------- ------------- 24.1/36.6 MB 22.7 MB/s eta 0:00:01
   ------------------

In [2]:
import sklearn

print(f"scikit-learn version: {sklearn.__version__}")

ImportError: DLL load failed while importing _trlib: An Application Control policy has blocked this file.

In [3]:
%pip uninstall scipy -y
%pip install scipy==1.16.3

Found existing installation: scipy 1.18.1
Uninstalling scipy-1.18.1:
  Successfully uninstalled scipy-1.18.1
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


   ---------------------------------------- 0.0/38.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/38.5 MB 3.8 MB/s eta 0:00:10
   --- ------------------------------------ 3.4/38.5 MB 7.3 MB/s eta 0:00:05
   ------ --------------------------------- 6.0/38.5 MB 9.1 MB/s eta 0:00:04
   --------- ------------------------------ 9.4/38.5 MB 10.8 MB/s eta 0:00:03
   ------------- -------------------------- 13.4/38.5 MB 12.3 MB/s eta 0:00:03
   ------------------ --------------------- 17.8/38.5 MB 13.9 MB/s eta 0:00:02
   ---------------------- ----------------- 21.8/38.5 MB 14.5 MB/s eta 0:00:02
   ----------------------------- ---------- 28.3/38.5 MB 16.6 MB/s eta 0:00:01
   ----------------------------------- ---- 34.6/38.5 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------  38.3/38.5 MB 19.1 MB/s eta 0:00:01
   ---------------------------------------- 38.5/38.5 MB 17.0 MB/s  0:00:02
N

In [2]:
import sys
import platform

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Windows:", platform.platform())
print("Architecture:", platform.architecture())

Python executable: C:\Users\abdal\SentinelFlow-AI\.venv\Scripts\python.exe
Python version: 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
Windows: Windows-11-10.0.26200-SP0
Architecture: ('64bit', 'WindowsPE')


In [3]:
import sklearn

In [4]:
import scipy
import sklearn

print("scipy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

scipy: 1.16.3
scikit-learn: 1.9.0


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

# Processed dataset location
PROCESSED_DIR = Path("data/processed")
processed_files = sorted(PROCESSED_DIR.glob("*.csv"))

# Final feature set selected during feature engineering
selected_features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "Packet Length Variance",
    "FIN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "ECE Flag Count",
    "Down/Up Ratio",
    "Average Packet Size",
    "Avg Fwd Segment Size",
    "Avg Bwd Segment Size",
    "Subflow Fwd Bytes",
    "Subflow Bwd Bytes",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "act_data_pkt_fwd",
    "min_seg_size_forward",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min"
]

print(f"Processed files: {len(processed_files)}")
print(f"Selected features: {len(selected_features)}")

Processed files: 8
Selected features: 65


In [6]:
# Check for duplicate ML rows across all processed files
seen_hashes = set()
cross_file_duplicates = 0
total_rows_checked = 0

columns_to_check = selected_features + ["Label"]

for file in processed_files:
    file_duplicates = 0

    for chunk in pd.read_csv(
        file,
        usecols=columns_to_check,
        chunksize=100000,
        low_memory=False
    ):
        row_hashes = pd.util.hash_pandas_object(
            chunk,
            index=False
        ).to_numpy()

        for row_hash in row_hashes:
            row_hash = int(row_hash)

            if row_hash in seen_hashes:
                cross_file_duplicates += 1
                file_duplicates += 1
            else:
                seen_hashes.add(row_hash)

        total_rows_checked += len(chunk)

    print(f"{file.name}: {file_duplicates:,} repeated rows")

print("\n--- Global Duplicate Audit ---")
print(f"Total rows checked: {total_rows_checked:,}")
print(f"Repeated rows found: {cross_file_duplicates:,}")
print(
    f"Repeated percentage: "
    f"{cross_file_duplicates / total_rows_checked * 100:.4f}%"
)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 0 repeated rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 1,523 repeated rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 3,947 repeated rows
Monday-WorkingHours.pcap_ISCX.csv: 186 repeated rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 6,934 repeated rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 7,600 repeated rows
Tuesday-WorkingHours.pcap_ISCX.csv: 11,643 repeated rows
Wednesday-workingHours.pcap_ISCX.csv: 20,009 repeated rows

--- Global Duplicate Audit ---
Total rows checked: 2,572,640
Repeated rows found: 51,842
Repeated percentage: 2.0151%


In [7]:
# Verify repeated rows and check for conflicting labels

seen_feature_hashes = {}
exact_repeats = 0
conflicting_labels = 0
total_rows_checked = 0

for file in processed_files:
    print(f"Checking: {file.name}")

    for chunk in pd.read_csv(
        file,
        usecols=selected_features + ["Label"],
        chunksize=100000,
        low_memory=False
    ):
        feature_hashes = pd.util.hash_pandas_object(
            chunk[selected_features],
            index=False
        ).to_numpy()

        labels = chunk["Label"].to_numpy()

        for row_hash, label in zip(feature_hashes, labels):
            row_hash = int(row_hash)

            if row_hash in seen_feature_hashes:
                if seen_feature_hashes[row_hash] == label:
                    exact_repeats += 1
                else:
                    conflicting_labels += 1
            else:
                seen_feature_hashes[row_hash] = label

        total_rows_checked += len(chunk)

print("\n--- Duplicate Verification ---")
print(f"Total rows checked: {total_rows_checked:,}")
print(f"Repeated features + same label: {exact_repeats:,}")
print(f"Repeated features + different label: {conflicting_labels:,}")

Checking: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Checking: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Checking: Friday-WorkingHours-Morning.pcap_ISCX.csv
Checking: Monday-WorkingHours.pcap_ISCX.csv
Checking: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Checking: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Checking: Tuesday-WorkingHours.pcap_ISCX.csv
Checking: Wednesday-workingHours.pcap_ISCX.csv

--- Duplicate Verification ---
Total rows checked: 2,572,640
Repeated features + same label: 51,842
Repeated features + different label: 697


In [8]:
# Analyze conflicting labels for identical feature vectors

feature_labels = {}
conflict_pairs = {}

for file in processed_files:
    print(f"Scanning: {file.name}")

    for chunk in pd.read_csv(
        file,
        usecols=selected_features + ["Label"],
        chunksize=100000,
        low_memory=False
    ):
        feature_hashes = pd.util.hash_pandas_object(
            chunk[selected_features],
            index=False
        ).to_numpy()

        labels = chunk["Label"].to_numpy()

        for row_hash, label in zip(feature_hashes, labels):
            row_hash = int(row_hash)

            if row_hash not in feature_labels:
                feature_labels[row_hash] = label

            elif feature_labels[row_hash] != label:
                pair = tuple(sorted([
                    str(feature_labels[row_hash]),
                    str(label)
                ]))

                conflict_pairs[pair] = conflict_pairs.get(pair, 0) + 1


print("\n--- Conflicting Label Pairs ---")

for pair, count in sorted(
    conflict_pairs.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{pair[0]}  <->  {pair[1]}: {count:,}")

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
Scanning: Monday-WorkingHours.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
Scanning: Wednesday-workingHours.pcap_ISCX.csv

--- Conflicting Label Pairs ---
BENIGN  <->  PortScan: 564
BENIGN  <->  DoS Hulk: 129
BENIGN  <->  DDoS: 3
BENIGN  <->  DoS slowloris: 1


In [9]:
# Build clean ML dataset:
# 1. Keep only selected features + Label
# 2. Remove exact duplicate rows
# 3. Remove feature vectors that have conflicting labels

ml_parts = []

for file in processed_files:
    print(f"Loading: {file.name}")

    part = pd.read_csv(
        file,
        usecols=selected_features + ["Label"],
        low_memory=False
    )

    ml_parts.append(part)

ml_df = pd.concat(
    ml_parts,
    ignore_index=True
)

original_rows = len(ml_df)

print(f"\nOriginal ML rows: {original_rows:,}")

# Remove exact duplicates
ml_df = ml_df.drop_duplicates(
    subset=selected_features + ["Label"],
    keep="first"
).reset_index(drop=True)

after_duplicates = len(ml_df)

print(
    f"Exact duplicate rows removed: "
    f"{original_rows - after_duplicates:,}"
)

# Identify feature vectors associated with more than one Label
feature_hash = pd.util.hash_pandas_object(
    ml_df[selected_features],
    index=False
)

ml_df["_feature_hash"] = feature_hash

labels_per_features = (
    ml_df.groupby("_feature_hash")["Label"]
    .nunique()
)

conflicting_hashes = labels_per_features[
    labels_per_features > 1
].index

conflicting_rows = ml_df[
    ml_df["_feature_hash"].isin(conflicting_hashes)
].shape[0]

print(
    f"Rows belonging to conflicting feature groups: "
    f"{conflicting_rows:,}"
)

# Remove all conflicting groups
ml_df = ml_df[
    ~ml_df["_feature_hash"].isin(conflicting_hashes)
].copy()

ml_df.drop(columns="_feature_hash", inplace=True)
ml_df.reset_index(drop=True, inplace=True)

print("\n--- Final ML Dataset ---")
print(f"Rows: {len(ml_df):,}")
print(f"Input features: {len(selected_features)}")
print(f"Columns including Label: {ml_df.shape[1]}")

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv

Original ML rows: 2,572,640
Exact duplicate rows removed: 51,842
Rows belonging to conflicting feature groups: 1,394

--- Final ML Dataset ---
Rows: 2,519,404
Input features: 65
Columns including Label: 66


In [10]:
# Create binary intrusion-detection target
# 0 = BENIGN
# 1 = ATTACK

ml_df["binary_target"] = (
    ml_df["Label"] != "BENIGN"
).astype("int8")

print("--- Binary Target Distribution ---")

binary_counts = ml_df["binary_target"].value_counts().sort_index()

for target, count in binary_counts.items():
    name = "BENIGN" if target == 0 else "ATTACK"
    percentage = count / len(ml_df) * 100

    print(
        f"{target} ({name}): "
        f"{count:,} rows ({percentage:.2f}%)"
    )

print(f"\nTotal rows: {len(ml_df):,}")

--- Binary Target Distribution ---
0 (BENIGN): 2,094,360 rows (83.13%)
1 (ATTACK): 425,044 rows (16.87%)

Total rows: 2,519,404


In [11]:
from sklearn.model_selection import train_test_split

# Row indices
all_indices = np.arange(len(ml_df))

# 70% Train / 30% Temporary
train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=42,
    stratify=ml_df["Label"]
)

# Split remaining 30% equally:
# 15% Validation / 15% Test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=ml_df.iloc[temp_idx]["Label"]
)

print("--- Dataset Split ---")
print(f"Train:      {len(train_idx):,} ({len(train_idx)/len(ml_df)*100:.2f}%)")
print(f"Validation: {len(val_idx):,} ({len(val_idx)/len(ml_df)*100:.2f}%)")
print(f"Test:       {len(test_idx):,} ({len(test_idx)/len(ml_df)*100:.2f}%)")

print(f"\nTotal: {len(train_idx) + len(val_idx) + len(test_idx):,}")

--- Dataset Split ---
Train:      1,763,582 (70.00%)
Validation: 377,911 (15.00%)
Test:       377,911 (15.00%)

Total: 2,519,404


In [12]:
# Verify class distribution across Train / Validation / Test

split_distribution = pd.DataFrame({
    "Total": ml_df["Label"].value_counts(),
    "Train": ml_df.iloc[train_idx]["Label"].value_counts(),
    "Validation": ml_df.iloc[val_idx]["Label"].value_counts(),
    "Test": ml_df.iloc[test_idx]["Label"].value_counts()
}).fillna(0).astype(int)

split_distribution["Train %"] = (
    split_distribution["Train"] / split_distribution["Total"] * 100
).round(2)

split_distribution["Validation %"] = (
    split_distribution["Validation"] / split_distribution["Total"] * 100
).round(2)

split_distribution["Test %"] = (
    split_distribution["Test"] / split_distribution["Total"] * 100
).round(2)

split_distribution

,Total,Train,Validation,Test,Train %,Validation %,Test %
Label,,,,,,,
BENIGN,2094360,1466051,314155,314154,70.00,15.00,15.00
DoS Hulk,172717,120902,25907,25908,70.00,15.00,15.00
DDoS,128011,89608,19202,19201,70.00,15.00,15.00
PortScan,90130,63091,13520,13519,70.00,15.00,15.00
DoS GoldenEye,10286,7200,1543,1543,70.00,15.00,15.00
FTP-Patator,5931,4152,889,890,70.01,14.99,15.01
DoS slowloris,5384,3769,807,808,70.00,14.99,15.01
DoS Slowhttptest,5228,3659,784,785,69.99,15.00,15.02
SSH-Patator,3219,2253,483,483,69.99,15.00,15.00


## Final ML Preparation Summary

The CIC-IDS2017 processed dataset was prepared for machine-learning development with the following final decisions:

- Total processed rows: 2,572,640
- Zero-variance features removed: 8
- Exact duplicate features removed: 5
- Final selected input features: 65
- Exact duplicate ML rows removed: 51,842
- Conflicting feature-label rows removed: 1,394
- Final ML dataset size: 2,519,404 rows

### Targets

Two prediction targets are retained:

- `binary_target`
  - 0 = BENIGN
  - 1 = ATTACK

- `Label`
  - Original 15-class attack label for multiclass classification

### Final Binary Distribution

- BENIGN: 2,094,360 rows (83.13%)
- ATTACK: 425,044 rows (16.87%)

### Dataset Split

A stratified split based on the original multiclass `Label` was used so that attack classes remain represented across all subsets.

- Training: 1,763,582 rows (70%)
- Validation: 377,911 rows (15%)
- Test: 377,911 rows (15%)

All 15 classes are represented in the training, validation, and test sets.

Rare classes such as Heartbleed, Infiltration, and Web Attack SQL Injection contain very few samples, so their per-class evaluation metrics must be interpreted cautiously.

The test set will remain untouched during model development and will only be used for final evaluation.

### Next Stage

The next notebook will begin model development.

Stage 1:
Binary intrusion detection — BENIGN vs ATTACK

Stage 2:
Multiclass attack classification for malicious traffic